# mahalanobis

A robust, multivariate alternative to `morphology_map.jl`'s GA-SaMMT Map step: instead of thresholding one derivative raster at `mean +/- C*std` (`threshold_mask`), build a per-cell vector across an arbitrary combination of derivative rasters and test it with a robust Mahalanobis distance at an explicit probability level `P`, per the user's own working draft (`Mahalanobis_Seabed_Morphology_Intro_Methodology.pdf`, Section 2.5-2.9). Reuses `polygonize_filtered` from `morphology_map.jl` unchanged for the final mask-to-polygon step; everything upstream of that is new.

`GA-SaMMT`'s own single-derivative `TPIMethod`/`OpennessMethod`/`BPIMethod` (`morphology_map.jl`) are untouched by this file -- they stay empty marker structs with radius passed as a separate argument, since that's all a single-derivative threshold needs. A *list* of derivatives needs each one to carry its own parameters (two `TPIMethod()` instances in a vector can't say which radius is which), so this file introduces a separate, permanent `ParameterizedMethod` hierarchy for the multi-derivative path rather than changing the existing one -- a deliberate decision, not a placeholder (see `project-mahalanobis-threshold` memory for the discussion that led to it).

In [ ]:
#| hide
using MIM_GA

In [ ]:
#| default_exp mahalanobis

In [ ]:
#| export
using Rasters
import ArchGDAL
using Statistics
using LinearAlgebra
using Distributions
using DataFrames

In [ ]:
#| export
"""
Marker type for the multi-derivative Mahalanobis path. Unlike `MorphologyMethod`'s
empty marker structs (`TPIMethod()`, `OpennessMethod()`, `BPIMethod()` -- radius
passed separately, fine for exactly one derivative), every `ParameterizedMethod`
carries its own parameters, so a `Vector{<:ParameterizedMethod}` can unambiguously
mix different radii and derivative types in one call -- e.g.
`[TPIParam(10), TPIParam(50), OpennessParam(50, :positive)]`.
"""
abstract type ParameterizedMethod end

"""TPI at `radius` (ground units, per `tpi`'s own `unit` keyword elsewhere -- here fixed to meters, see `resolve_derivative`)."""
struct TPIParam <: ParameterizedMethod
    radius::Float64
end

"""BPI with the given inner/outer annulus radii."""
struct BPIParam <: ParameterizedMethod
    inner_radius::Float64
    outer_radius::Float64
end

"""
Openness at `radius`, explicitly `sign = :positive` or `:negative`. GA-SaMMT's own
`OpennessMethod` (`morphology_map.jl`) uses Negative Openness for highs and Positive
for lows -- an asymmetric convention baked into that file's two methods. A
multi-derivative stack has no single built-in notion of "high"/"low" for Openness
(the Mahalanobis sign-gating rule in `morphological_classify` is the same `>0`/`<0`
test for every axis), so the caller must pick PO or NO explicitly here rather than
it being an implicit, undocumented default.
"""
struct OpennessParam <: ParameterizedMethod
    radius::Float64
    sign::Symbol
    function OpennessParam(radius::Real, sign::Symbol)
        sign in (:positive, :negative) || throw(ArgumentError("sign must be :positive or :negative, got :$sign"))
        new(radius, sign)
    end
end

In [ ]:
#| export
"""
    resolve_derivative(method::ParameterizedMethod, bathy::Bathymetry) -> (Raster, Symbol)

Compute one `ParameterizedMethod`'s derivative raster and a unique name for it
(used as the resulting `RasterStack`'s layer key). One method per concrete
`ParameterizedMethod` subtype, each calling the matching `morphometrics.jl`
function and resolving its own algorithm's convention (e.g. which Openness sign).
"""
resolve_derivative(m::TPIParam, bathy::Bathymetry) = (tpi(bathy, m.radius).raster, Symbol("tpi_$(Int(m.radius))"))

resolve_derivative(m::BPIParam, bathy::Bathymetry) = (bpi(bathy, m.inner_radius, m.outer_radius).raster,
                                                       Symbol("bpi_$(Int(m.inner_radius))_$(Int(m.outer_radius))"))

function resolve_derivative(m::OpennessParam, bathy::Bathymetry)
    o = openness(bathy, m.radius)
    r = m.sign == :positive ? o.positive.raster : o.negative.raster
    return r, Symbol("openness_$(m.sign)_$(Int(m.radius))")
end

In [ ]:
#| export
"""
    derivative_stack(methods::Vector{<:ParameterizedMethod}, bathy::Bathymetry) -> RasterStack

Resolve every method to its derivative raster and combine them into one
`RasterStack` -- the per-cell vector `x(i,j)` from the paper's Section 2.3,
generalized from a fixed `[TPI_r1, TPI_r2]` to any combination/count of
derivatives. Errors if two methods resolve to the same layer name (e.g. two
identical `TPIParam`s) rather than silently dropping one.
"""
function derivative_stack(methods::Vector{<:ParameterizedMethod}, bathy::Bathymetry)
    length(methods) >= 1 || throw(ArgumentError("need at least one method"))
    pairs = [resolve_derivative(m, bathy) for m in methods]
    names = Tuple(n for (_, n) in pairs)
    length(unique(names)) == length(names) ||
        throw(ArgumentError("duplicate derivative names in $names -- two identical methods? disambiguate (e.g. different radii)"))
    rasters = Tuple(r for (r, _) in pairs)
    return RasterStack(NamedTuple{names}(rasters))
end

In [ ]:
#| export
"""
    robust_mu_sigma(X::AbstractMatrix, p::Int; alpha=0.025, maxiter=50, tol=1e-10) -> (mu, Sigma, niter)

Robust location/scatter estimate for the paper's Section 2.6: iteratively trim to
the `h = floor(n*(1-alpha))` cells with the *smallest* Mahalanobis distance under
the current `(mu, Sigma)`, re-estimate from that fixed-size subset, rescale by the
chi-squared consistency factor `c_alpha` (Eq. 7), repeat to convergence. This is
the computationally light alternative to full FAST-MCD the paper names as a
fallback (true FAST-MCD has no ready Julia package as of this writing).

**Trims to a fixed count, not a fixed threshold -- this is a deliberate departure
from the paper's literal Eq. 7 wording, not an oversight.** Implemented literally
(`H = {cells with D^2 <= chi^2_{p,1-alpha}}`, evaluated against the *current*,
iteration-varying `Sigma`), the procedure is unstable: a smaller `Sigma` admits
fewer cells under the same fixed threshold on the next round, which shrinks `Sigma`
further, collapsing to a degenerate estimate (verified: 44.7% of cells flagged at
P=0.95 on a real tile, vs 6.6% classical -- not a plausible foreground/background
split). Trimming to a *fixed count* of smallest-D^2 cells each round removes that
feedback loop and is structurally what FAST-MCD's own concentration ("C-") steps
do. `c_alpha` is applied every iteration, not just once at the end -- skipping the
per-iteration rescale reproduces the same instability.
"""
function robust_mu_sigma(X::AbstractMatrix, p::Int; alpha::Real=0.025, maxiter::Integer=50, tol::Real=1e-10)
    n = size(X, 1)
    h = floor(Int, n * (1 - alpha))
    mu = vec(mean(X, dims=1))
    Sigma = cov(X)
    chi2_thresh = quantile(Chisq(p), 1 - alpha)
    c_alpha = (1 - alpha) / cdf(Chisq(p + 2), chi2_thresh)

    niter = 0
    for iter in 1:maxiter
        niter = iter
        L = cholesky(Sigma).L
        Xc = X .- mu'
        Z = (L \ Xc')'
        D2 = vec(sum(Z .^ 2, dims=2))
        H = partialsortperm(D2, 1:h)

        mu_new = vec(mean(view(X, H, :), dims=1))
        Sigma_new = c_alpha .* cov(view(X, H, :))

        delta = norm(mu_new .- mu) + norm(Sigma_new .- Sigma)
        mu, Sigma = mu_new, Sigma_new
        delta < tol && break
    end

    return mu, Sigma, niter
end

In [ ]:
#| export
"""
    morphological_classify(stack::RasterStack; alpha=0.025, P=0.95) -> NamedTuple

The robust multivariate Map-step test (paper Sections 2.5-2.8): flatten `stack` to
the valid-cell matrix `X` (cells non-NaN in *every* layer), estimate `(mu, Sigma)`
robustly (`robust_mu_sigma`), compute the squared Mahalanobis distance `D^2` via
Cholesky whitening, threshold at `tau = chi^2_p(P)`, then directionally gate every
flagged cell into `:high` (every axis centred-positive), `:low` (every axis
centred-negative) or `:mixed` (axes disagree -- nested morphology, no GA-SaMMT
counterpart). The `p=2` directional rule is the paper's own Table 1; `all(>(0),
row)`/`all(<(0), row)` is the natural generalisation to any `p`, which the paper
does not itself state for `p>2`.

Returns a `NamedTuple` (`idx`, `class`, `D2`, `Xc`, `mu_robust`, `Sigma_robust`,
`niter`, `tau`, `layer_names`, `dims`) rather than just a mask -- `mu_robust`/
`Sigma_robust`/`niter`/`D2`/`tau` are useful on their own for diagnostics (e.g. the
mean `D^2` per polygon the paper's Characterise step wants as a confidence index),
and `Xc` (the robustly-centred data matrix, one column per `layer_names` entry) is
what `regional_polygons` uses for the containment sub-step (1g) -- not just
intermediate values.
"""
function morphological_classify(stack::RasterStack; alpha::Real=0.025, P::Real=0.95)
    layer_names = collect(keys(stack))
    mats = [Array(replace_missing(stack[name], NaN)) for name in layer_names]
    valid = reduce((a, b) -> a .& .!isnan.(b), mats; init=trues(size(mats[1])))
    idx = findall(valid)
    X = hcat((m[idx] for m in mats)...)
    p = size(X, 2)

    mu_robust, Sigma_robust, niter = robust_mu_sigma(X, p; alpha=alpha)

    L = cholesky(Sigma_robust).L
    Xc = X .- mu_robust'
    Z = (L \ Xc')'
    D2 = vec(sum(Z .^ 2, dims=2))
    tau = quantile(Chisq(p), P)
    flagged = D2 .> tau

    class = fill(:normal, length(D2))
    @inbounds for i in eachindex(class)
        flagged[i] || continue
        row = view(Xc, i, :)
        if all(>(0), row)
            class[i] = :high
        elseif all(<(0), row)
            class[i] = :low
        else
            class[i] = :mixed
        end
    end

    return (idx=idx, class=class, D2=D2, Xc=Xc, mu_robust=mu_robust, Sigma_robust=Sigma_robust,
            niter=niter, tau=tau, layer_names=layer_names, dims=size(mats[1]))
end

In [ ]:
#| exporti
# Scatter morphological_classify's flat `class` vector back into a 2D mask
# for one label, matching `stack`'s own raster shape.
function morphological_mask(result, label::Symbol)
    mask = fill(false, result.dims)
    for (i, c) in zip(result.idx, result.class)
        c == label && (mask[i] = true)
    end
    return mask
end

In [ ]:
#| export
"""
    regional_polygons(stack::RasterStack, result, layer::Symbol) -> DataFrame

The large-scale-only polygon set from the paper's containment sub-step (Section
2.9, 1g): every valid cell split into `"regional_high"`/`"regional_low"` purely
by the *sign* of one chosen derivative's robustly-centred value (`layer`, one of
`result.layer_names`, taken from `morphological_classify`'s `Xc`) -- not gated by
the `D^2` flag at all, unlike the high/low/mixed classes `morphological_classify`
itself produces. This is the reference set `assign_containment!` assigns feature
polygons against.

The paper's own worked example uses `Delta_r2` (the larger of two TPI radii) for
this; the generalised API has no single designated "regional" axis in an
arbitrary derivative stack, so the caller names which layer plays that role
explicitly, rather than this guessing (e.g. "last layer in the stack").
`:parent_id`/`:parent_class` columns are added to the returned `DataFrame`
(`parent_id` unique per row, `parent_class` either `"regional_high"` or
`"regional_low"`).
"""
function regional_polygons(stack::RasterStack, result, layer::Symbol)
    k = findfirst(==(layer), result.layer_names)
    k === nothing && throw(ArgumentError("layer :$layer not in $(result.layer_names)"))
    delta = view(result.Xc, :, k)

    high_mat = fill(false, result.dims)
    low_mat  = fill(false, result.dims)
    for (i, d) in zip(result.idx, delta)
        if d > 0
            high_mat[i] = true
        elseif d < 0
            low_mat[i] = true
        end
    end

    template = stack[result.layer_names[1]]
    Q_high = polygonize_filtered(high_mat, template, 0.0)
    Q_low  = polygonize_filtered(low_mat, template, 0.0)
    Q_high.parent_class .= "regional_high"
    Q_low.parent_class  .= "regional_low"
    Q_all = vcat(Q_high, Q_low)
    Q_all.parent_id = 1:nrow(Q_all)
    return Q_all
end

In [ ]:
#| exporti
# Best (geometry, area) match for one feature polygon against the regional set Q,
# by max areal overlap fraction |q ∩ Q| / |q|; (missing, missing) if nothing clears
# `threshold`. `ArchGDAL.intersects` is a cheap pre-filter before the actual
# `intersection`/`geomarea` call, skipping pairs that can't overlap at all.
function _best_overlap_parent(q_geom, Q::DataFrame, threshold::Real)
    best_id = missing
    best_class = missing
    best_frac = 0.0
    q_area = ArchGDAL.geomarea(q_geom)
    for row in eachrow(Q)
        ArchGDAL.intersects(q_geom, row.geometry) || continue
        inter = ArchGDAL.intersection(q_geom, row.geometry)
        frac = ArchGDAL.geomarea(inter) / q_area
        if frac > best_frac
            best_frac = frac
            best_id = row.parent_id
            best_class = row.parent_class
        end
    end
    return best_frac >= threshold ? (best_id, best_class) : (missing, missing)
end

In [ ]:
#| export
"""
    assign_containment!(features::DataFrame, Q::DataFrame; threshold=0.5) -> DataFrame

Assign each row of `features` (a feature-polygon `DataFrame` from
`morphological_features`) a `:parent_id`/`:parent_class` from the large-scale-only
polygon set `Q` (`regional_polygons`), by maximum areal overlap `argmax |q ∩ Q| /
|q|`, kept only if that fraction is `>= threshold`; otherwise `missing`
(unparented -- open, unstructured seabed, per the paper). Brute-force over all
`(q, Q)` pairs (cheap `ArchGDAL.intersects` pre-filter first); fine while `Q`
stays in the low thousands, would want a spatial index (e.g. an R-tree) at much
larger scale. Mutates `features` in place (adds the two columns) and returns it.

`morphological_features` already calls this internally when its own
`containment_layer` keyword is given -- call it directly only if you want to
apply containment separately (e.g. a different `threshold`, or against a
different `Q`) without re-running the classification.
"""
function assign_containment!(features::DataFrame, Q::DataFrame; threshold::Real=0.5)
    parent_ids = Vector{Union{Missing,Int}}(undef, nrow(features))
    parent_classes = Vector{Union{Missing,String}}(undef, nrow(features))
    for (i, row) in enumerate(eachrow(features))
        pid, pclass = _best_overlap_parent(row.geometry, Q, threshold)
        parent_ids[i] = pid
        parent_classes[i] = pclass
    end
    features.parent_id = parent_ids
    features.parent_class = parent_classes
    return features
end

In [ ]:
#| export
"""
    morphological_features(methods::Vector{<:ParameterizedMethod}, bathy::Bathymetry;
                            alpha=0.025, P=0.95, area_threshold=0.0,
                            containment_layer::Union{Symbol,Nothing}=nothing,
                            containment_threshold=0.5) -> (DataFrame, NamedTuple)

The robust multivariate Map step's single entry point: one `derivative_stack` and
one `morphological_classify` call, high/low/mixed polygonised and `vcat`'d into a
single `DataFrame` tagged by a `:kind` column (`"high"`/`"low"`/`"mixed"`) --
exactly the `:geometry`/`:area`/`:kind` schema the Lab prototype notebook built by
hand. Everything derives from the same `result`, so there is exactly one robust
fit per call -- no risk of the independent-call inconsistency an earlier version
of this file had (separate `morphological_high`/`low`/`mixed` functions, each
re-running `derivative_stack`/`robust_mu_sigma` from scratch: wasteful, and not
even guaranteed to agree with each other, since `robust_mu_sigma` is an iterative
fixed-point solve and separate calls can converge to `mu_robust`/`Sigma_robust`
that differ at the level of floating-point noise -- observed, on a real tile, as
~134 cells out of 7579 landing on the wrong side of a containment boundary.
Removed rather than left as a documented footgun).

Pass `containment_layer` (one of the stack's layer names, e.g. `:tpi_50`) to also
run the containment sub-step (1g) in the same call: `regional_polygons` plus
`assign_containment!` against the returned `features`, adding `:parent_id`/
`:parent_class` columns. Omitted (the default) skips containment entirely.

Returns `(features, result)` -- `result` (`morphological_classify`'s
`NamedTuple`) is also what `regional_polygons` needs if you want to call
containment separately later instead of via `containment_layer` here.
"""
function morphological_features(methods::Vector{<:ParameterizedMethod}, bathy::Bathymetry;
                                 alpha::Real=0.025, P::Real=0.95, area_threshold::Real=0.0,
                                 containment_layer::Union{Symbol,Nothing}=nothing,
                                 containment_threshold::Real=0.5)
    stack = derivative_stack(methods, bathy)
    result = morphological_classify(stack; alpha=alpha, P=P)
    template = stack[result.layer_names[1]]

    high  = polygonize_filtered(morphological_mask(result, :high),  template, area_threshold)
    low   = polygonize_filtered(morphological_mask(result, :low),   template, area_threshold)
    mixed = polygonize_filtered(morphological_mask(result, :mixed), template, area_threshold)
    high.kind  .= "high"
    low.kind   .= "low"
    mixed.kind .= "mixed"
    features = vcat(high, low, mixed)

    if containment_layer !== nothing
        Q = regional_polygons(stack, result, containment_layer)
        assign_containment!(features, Q; threshold=containment_threshold)
    end

    return features, result
end